# Run Evaluation, Test Trajectory Visualization & SMILES Inference

Loads a saved training run from `./runs/...`, plots training and validation metrics, visualizes reverse-diffusion coordinate trajectories on held-out test molecules (`data/test.parquet`), and generates 3D conformers directly from SMILES strings.

In [ ]:
import json
import pathlib
import tempfile
from typing import Final

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from miniqm import viz
from miniqm.data import batch as batch_lib
from miniqm.data import data_source
from miniqm.diffusion import geometry, losses, model, sampling
from miniqm.training import checkpoint, train
from miniqm.training import eval as eval_lib

sns.set_theme(style="whitegrid")

REPO_ROOT: Final = (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
)
DEFAULT_RUN_DIR: Final = REPO_ROOT / "runs" / "qm9_experiment"
TEST_PARQUET: Final = REPO_ROOT / "data" / "test.parquet"
BATCH_SIZE: Final = 32

try:
    run_dir = checkpoint.resolve_run_dir(DEFAULT_RUN_DIR)
except FileNotFoundError:
    demo_parent = pathlib.Path(tempfile.mkdtemp(prefix="miniqm_demo_run_"))
    print(f"No saved run at {DEFAULT_RUN_DIR}; running a 20-step demo job in {demo_parent}...")
    train.run_training(
        train.RunConfig(
            train_path=REPO_ROOT / "data" / "train.parquet",
            val_path=REPO_ROOT / "data" / "validation.parquet",
            run_dir=demo_parent,
            batch_size=16,
            num_steps=20,
            log_every=5,
            eval_every=10,
            sample_every=10,
            max_train_molecules=64,
            max_val_molecules=32,
            sample_batch_size=8,
            train=train.TrainConfig(
                model=model.DenoiserConfig(
                    transformer=model.TransformerConfig(
                        hidden_dim=64,
                        num_layers=2,
                        num_heads=4,
                    ),
                ),
                learning_rate=1e-3,
                warmup_steps=5,
                decay_steps=20,
                ema_decay=0.9,
            ),
            sampling=sampling.SamplingConfig(num_steps=20),
        )
    )
    run_dir = checkpoint.resolve_run_dir(demo_parent)

loaded_run = checkpoint.load_run_checkpoint(run_dir, checkpoint_name="best.msgpack")
print(
    f"Loaded checkpoint from {loaded_run.run_dir} at step={loaded_run.checkpoint.step} "
    f"({train.count_parameters(loaded_run.checkpoint.ema_params):,} parameters)."
)

In [ ]:
metrics_path = run_dir / "metrics.jsonl"
records = [
    json.loads(line)
    for line in metrics_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
metrics_df = pd.DataFrame(records)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

loss_df = metrics_df[metrics_df["split"].isin(["train", "val"])]
sns.lineplot(
    data=loss_df,
    x="step",
    y="total_loss",
    hue="split",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Diffusion Loss vs. Step")
axes[0].set_xlabel("Optimization Step")
axes[0].set_ylabel("Total Diffusion Loss")

sns.lineplot(
    data=loss_df,
    x="step",
    y="rmsd",
    hue="split",
    marker="o",
    ax=axes[1],
)
sample_df = metrics_df[metrics_df["split"] == "val_sample"]
if not sample_df.empty:
    axes[1].plot(
        sample_df["step"],
        sample_df["sampled_rmsd_mean"],
        marker="s",
        linestyle="--",
        color="tab:green",
        label="val_sample (reverse ODE)",
    )
    axes[1].legend()
axes[1].set_title("Coordinate RMSD (Å) vs. Step")
axes[1].set_xlabel("Optimization Step")
axes[1].set_ylabel("Kabsch RMSD (Å)")

plt.tight_layout()
plt.show()

In [ ]:
test_source = data_source.DataSource(TEST_PARQUET, max_molecules=BATCH_SIZE)
test_loss = eval_lib.evaluate_loss(
    loaded_run.checkpoint.ema_params,
    jax.random.key(0),
    test_source,
    batch_size=8,
    model_config=loaded_run.model_config,
    loss_config=loaded_run.loss_config,
)
test_sampling = eval_lib.evaluate_dataset_sampling(
    loaded_run.checkpoint.ema_params,
    jax.random.key(1),
    test_source,
    batch_size=8,
    model_config=loaded_run.model_config,
    sampling_config=loaded_run.sampling_config,
)
print(
    f"Test split metrics ({len(test_source)} molecules): "
    f"loss={test_loss.total_loss:.4f}, teacher_forced_rmsd={test_loss.rmsd:.3f} Å, "
    f"sampled_rmsd_mean={test_sampling.sampled_rmsd_mean:.3f} Å, "
    f"sampled_lddt_mean={test_sampling.sampled_lddt_mean:.3f}"
)

host_test_batch = next(test_source.iter_batches(batch_size=BATCH_SIZE, drop_last=False))
jax_test_batch = host_test_batch.to_jax()
denoiser = model.Denoiser(config=loaded_run.model_config)
test_trajectory = sampling.sample_trajectory(
    jax.random.key(42),
    denoiser,
    loaded_run.checkpoint.ema_params,
    batch=jax_test_batch,
    config=loaded_run.sampling_config,
)
generated_test_positions = test_trajectory[-1]
aligned_test_positions = geometry.kabsch_align(
    reference=jax_test_batch.positions,
    target=generated_test_positions,
    atom_mask=jax_test_batch.atom_mask,
)
atom_counts = jnp.maximum(jnp.sum(jax_test_batch.atom_mask, axis=-1), 1)
per_mol_rmsd = jnp.sqrt(
    jnp.sum(jnp.square(aligned_test_positions - jax_test_batch.positions), axis=(-1, -2))
    / atom_counts
)
per_mol_lddt = losses.lddt_per_molecule(
    predicted_positions=generated_test_positions,
    target_positions=jax_test_batch.positions,
    atom_mask=jax_test_batch.atom_mask,
)

In [ ]:
selected_idx = np.random.default_rng().integers(0, BATCH_SIZE)
mol_rmsd = float(per_mol_rmsd[selected_idx])
mol_lddt = float(per_mol_lddt[selected_idx])
print(
    f"Test molecule {selected_idx}: Kabsch RMSD = {mol_rmsd:.3f} Å, lDDT = {mol_lddt:.3f} "
    "(translucent = ground truth, cyan = generated conformer)"
)
overlay_viewer = viz.show_overlay(
    reference_positions=np.asarray(jax_test_batch.positions[selected_idx]),
    candidate_positions=np.asarray(aligned_test_positions[selected_idx]),
    atom_features=host_test_batch.atom_features[selected_idx],
    atom_mask=host_test_batch.atom_mask[selected_idx],
)
overlay_viewer.show()

In [ ]:
print(
    f"Animated reverse-diffusion trajectory ({test_trajectory.shape[0]} frames) "
    f"for test molecule {selected_idx}"
)
trajectory_viewer = viz.show_trajectory(
    trajectory=np.asarray(test_trajectory[:, selected_idx]),
    atom_features=host_test_batch.atom_features[selected_idx],
    atom_mask=host_test_batch.atom_mask[selected_idx],
    interval_ms=80,
)
trajectory_viewer.show()

In [ ]:
CUSTOM_SMILES: Final = ("CCO", "c1ccccc1", "CC(=O)O")
smiles_batch = batch_lib.collate_smiles(CUSTOM_SMILES)
smiles_trajectory = sampling.sample_trajectory(
    jax.random.key(7),
    denoiser,
    loaded_run.checkpoint.ema_params,
    batch=smiles_batch.to_jax(),
    config=loaded_run.sampling_config,
)

smiles_idx = 1
print(
    f"Generated 3D diffusion trajectory for SMILES={CUSTOM_SMILES[smiles_idx]!r} "
    f"({int(smiles_batch.atom_mask[smiles_idx].sum())} atoms)"
)
smiles_viewer = viz.show_trajectory(
    trajectory=np.asarray(smiles_trajectory[:, smiles_idx]),
    atom_features=smiles_batch.atom_features[smiles_idx],
    atom_mask=smiles_batch.atom_mask[smiles_idx],
    interval_ms=80,
)
smiles_viewer.show()